# Simulador Cache 4-Way no Colab

Este notebook executa o projeto público [Simulador-Cache-4-Way](https://github.com/LucasMGcode/Simulador-Cache-4-Way) no Google Colab.

A ideia é clonar o repositório, instalar o Icarus Verilog e rodar o testbench auto-verificável com `make sim`.

**English summary:** this notebook clones the public repository, installs Icarus Verilog, and runs the self-checking 4-way cache simulation.


## Material de apoio

- [README do projeto](https://github.com/LucasMGcode/Simulador-Cache-4-Way)
- [SVG do datapath](https://github.com/LucasMGcode/Simulador-Cache-4-Way/blob/main/assets/cache4way_datapath.svg)
- [Datapath da cache](https://github.com/LucasMGcode/Simulador-Cache-4-Way/blob/main/docs/datapath.md)
- [Desenho Inkscape e marcadores](https://github.com/LucasMGcode/Simulador-Cache-4-Way/blob/main/docs/inkscape-drawing.md)
- [FSM principal](https://github.com/LucasMGcode/Simulador-Cache-4-Way/blob/main/docs/fsm.md)
- [Política LRU](https://github.com/LucasMGcode/Simulador-Cache-4-Way/blob/main/docs/lru.md)
- [Roteiro de apresentação](https://github.com/LucasMGcode/Simulador-Cache-4-Way/blob/main/docs/presentation.md)


In [ ]:
# Instala o Icarus Verilog no ambiente do Colab.
!apt-get update -qq
!apt-get install -y -qq iverilog
!iverilog -V | head -n 3


In [ ]:
# Clona a versão pública mais recente do projeto.
!rm -rf Simulador-Cache-4-Way
!git clone https://github.com/LucasMGcode/Simulador-Cache-4-Way.git
%cd Simulador-Cache-4-Way
!git log --oneline -1


In [ ]:
# Executa a simulação e o testbench auto-verificável.
%cd /content/Simulador-Cache-4-Way/src
!make sim


## Visualização do datapath

A próxima célula exibe o SVG autoral do datapath. Ele é editável no Inkscape e contém marcadores `@...` que serão substituídos com os dados gerados pela simulação.

**English summary:** the next cells render the Inkscape-editable SVG and fill its dynamic markers using the simulation trace.


In [ ]:
from pathlib import Path
from IPython.display import HTML, display

REPO = Path("/content/Simulador-Cache-4-Way")
SVG_PATH = REPO / "assets" / "cache4way_datapath.svg"

def display_responsive_svg(svg_text):
    display(HTML(f"""
    <div style="width:100%; overflow-x:auto; padding: 8px 0;">
      <div style="min-width: 1120px; max-width: 1600px; margin: 0 auto;">
        {svg_text.replace('<svg', '<svg style="width:100%; height:auto; display:block;"', 1)}
      </div>
    </div>
    """))

display_responsive_svg(SVG_PATH.read_text(encoding="utf-8"))


In [ ]:
import csv
import html
import re
from pathlib import Path

import ipywidgets as widgets
from IPython.display import HTML, clear_output, display

REPO = Path("/content/Simulador-Cache-4-Way")
SVG_PATH = REPO / "assets" / "cache4way_datapath.svg"
TRACE_PATH = REPO / "src" / "trace.csv"

template_svg = SVG_PATH.read_text(encoding="utf-8")
rows = list(csv.DictReader(TRACE_PATH.open(encoding="utf-8")))

STATE_NAMES = {
    "0": "COMPARE",
    "1": "HIT",
    "2": "MISS",
    "3": "FILL_BLOCK",
    "4": "UPDATE_TAG",
}

def event_class(event):
    if event == "hit" or event.startswith("hit-"):
        return "hit"
    if "replace" in event:
        return "replace"
    return "miss"

def responsive_svg_html(svg_text):
    svg_text = svg_text.replace('<svg', '<svg style="width:100%; height:auto; display:block;"', 1)
    return f"""
    <div style="width:100%; overflow-x:auto; padding: 8px 0 4px;">
      <div style="min-width: 1120px; max-width: 1600px; margin: 0 auto;">
        {svg_text}
      </div>
    </div>
    """

def render_svg(row):
    addr = int(row["addr"])
    tag = int(row["tag"])
    line = int(row["line"])
    blk = int(row["blk"])
    replacements = {
        "@addr": row["addr"],
        "@addr_bin": f"{addr:012b}",
        "@tag": row["tag"],
        "@tag_bin": f"{tag:08b}",
        "@line": row["line"],
        "@line_bin": f"{line:02b}",
        "@blk": row["blk"],
        "@blk_bin": f"{blk:02b}",
        "@hit": row["hit"],
        "@selected_way": row["selected_way"],
        "@dout": row["dout"],
        "@state": STATE_NAMES.get(row["state"], row["state"]),
        "@event": row["event"],
    }
    for i in range(4):
        replacements[f"@valid{i}"] = row[f"valid{i}"]
        replacements[f"@tag{i}"] = row[f"tag{i}"]
        replacements[f"@lru{i}"] = row[f"lru{i}"]

    svg = template_svg
    for marker, value in sorted(replacements.items(), key=lambda item: len(item[0]), reverse=True):
        svg = svg.replace(marker, html.escape(str(value)))

    selected = row["selected_way"]
    svg = svg.replace(
        f'id="way{selected}-card" class="way-card"',
        f'id="way{selected}-card" class="way-card selected"',
    )
    svg = svg.replace(
        'id="event-badge" class="event-badge"',
        f'id="event-badge" class="event-badge {event_class(row["event"])}"',
    )
    remaining = sorted(set(re.findall(r"@\w+", svg)))
    if remaining:
        raise ValueError(f"Marcadores sem substituição: {remaining}")
    return svg

def row_log(row):
    state = STATE_NAMES.get(row["state"], row["state"])
    cells = [
        ("Passo", row["step"]),
        ("Cenário", row["label"]),
        ("Evento", row["event"]),
        ("Endereço", row["addr"]),
        ("Tag / Line / Blk", f'{row["tag"]} / {row["line"]} / {row["blk"]}'),
        ("Hit", row["hit"]),
        ("Via", row["selected_way"]),
        ("Dout", row["dout"]),
        ("FSM", state),
        ("LRU", f'{row["lru0"]}, {row["lru1"]}, {row["lru2"]}, {row["lru3"]}'),
    ]
    cards = "".join(
        f"""<div style="background:rgba(255,255,255,.82); border:1px solid #dbe4ee;
                    border-radius:14px; padding:10px 12px; box-shadow:0 8px 20px rgba(15,23,42,.08);
                    min-width:0; overflow:hidden;">
              <div style="font:600 11px Georgia,serif; color:#64748b;">{html.escape(name)}</div>
              <div style="font:800 15px ui-monospace,Menlo,Consolas,monospace; color:#0f172a;
                          line-height:1.25; white-space:normal; overflow-wrap:anywhere; word-break:break-word;">{html.escape(str(value))}</div>
            </div>"""
        for name, value in cells
    )
    return f"""
    <div style="max-width:1600px; margin:10px auto 0; padding:14px; border-radius:20px;
                background:linear-gradient(135deg, rgba(248,251,255,.92), rgba(239,246,255,.72));
                border:1px solid #dbe4ee; box-shadow:0 12px 28px rgba(15,23,42,.10);">
      <div style="font:800 16px Georgia,serif; color:#1e293b; margin-bottom:10px;">Log do acesso selecionado</div>
      <div style="display:grid; grid-template-columns:repeat(auto-fit,minmax(170px,1fr)); gap:10px;">{cards}</div>
    </div>
    """

slider = widgets.IntSlider(value=1, min=1, max=len(rows), step=1, description="Passo")
prev_button = widgets.Button(description="Anterior")
next_button = widgets.Button(description="Próximo")
output = widgets.Output()

def show_step(step):
    row = rows[step - 1]
    with output:
        clear_output(wait=True)
        display(HTML(responsive_svg_html(render_svg(row))))
        display(HTML(row_log(row)))

def on_slider_change(change):
    if change["name"] == "value":
        show_step(change["new"])

def previous(_):
    slider.value = max(slider.min, slider.value - 1)

def next_(_):
    slider.value = min(slider.max, slider.value + 1)

prev_button.on_click(previous)
next_button.on_click(next_)
slider.observe(on_slider_change, names="value")

display(widgets.HBox([prev_button, next_button]), slider, output)
show_step(slider.value)


## Como interpretar a saída

O testbench imprime cada acesso à cache, valida os sinais esperados e gera `trace.csv` para a visualização.

- `ACCESS`: mostra o cenário testado, endereço, tag, linha, bloco, hit, via selecionada, dado e LRU.
- `PASS`: indica que o valor observado bateu com o valor esperado.
- `FAIL`: indica erro; o testbench chama `$fatal(1)` e a simulação termina com falha.
- `ALL TESTS PASSED`: indica que todos os cenários principais passaram.
- `trace.csv`: alimenta o desenho SVG, substituindo os marcadores `@...` por sinais reais do acesso selecionado.

Cenários cobertos: miss com via inválida, hit após carregamento, substituição com todas as vias válidas, atualização da política LRU, acessos a diferentes linhas da cache e offsets diferentes dentro do mesmo bloco.
